# Données du tableau de bord RSE

In [ ]:
# Fabric notebook: nb_consolidate_data_for_CSR_report
# Workspace: BM_F_D - SAP-B1 / Lakehouse: LH_CSR_Reporting
#
# À lancer juste après nb_consolidate_csr_data, dans le même pipeline.
# Prépare les deux tables lues par le tableau de bord RSE :
#
#   - CSR_gold_reporting : les indicateurs de CSR_indicators_report + les
#     ventes, en format long (Entity, BU, Year, Month, MonthName, ID, Value,
#     Date). Les ventes viennent de la mesure "Sales | Commercial" du modèle
#     sémantique du dashboard commercial (scénario 1 = réel, même chiffre que
#     le rapport commercial), une ligne par entité SAP et par mois :
#       ID = "Env.1"   -> ventes en k€
#       ID = "Sales_€" -> ventes en €
#     Les indicateurs RSE n'ont pas d'entité (Entity vide).
#   - CSR_gold_tracking : CSR_completion_report (% de complétion par BU et
#     par mois) avec Year, Month (numéro), MonthName et Date.
#
# Date = 1er du mois, pour la relation avec d_Calendar.
#
# Aucun indicateur n'est calculé ici : les ratios (ex. Wat.2 = Wat.1 / Env.1)
# sont calculés à la volée dans le tableau de bord, au niveau d'agrégation
# affiché.
#
# Versionné dans le dépôt GitHub CSR_reporting (scripts/), exporté depuis
# Fabric sans les sorties (import-fabric-notebook, paquet bioline_utils).

In [ ]:
import pandas as pd
import sempy.fabric as fabric
from pyspark.sql.types import DateType, DoubleType, LongType, StringType, StructField, StructType

In [ ]:
# Paramètres
WS = "BM_F_D - SAP-B1"
DS = "Bioline Agrosciences sales - Copy"  # modèle sémantique du dashboard commercial
SALES_MEASURE = "Sales | Commercial"
FIRST_YEAR = 2026  # première année du reporting RSE

REPORTING_TABLE = "CSR_gold_reporting"
TRACKING_TABLE = "CSR_gold_tracking"

MONTHS = ["January", "February", "March", "April", "May", "June", "July",
          "August", "September", "October", "November", "December"]

# Entité SAP (d_Entity[Entity]) -> BU du reporting RSE. Plusieurs entités
# peuvent aller dans la même BU : le tableau de bord additionne par BU.
ENTITY_TO_BU = {
    "BIOLINE AGROSCIENCES FRANCE": "BFR",
    "VIRIDAXIS": "Viridaxis",
    "BIOLINE UK": "BUK",
    "DUDUTECH IPM LTD": "BAF",
    "DUDUTECH PROPERTIES LTD": "BAF",
    "BIOLINE IBERIA": "BIB",
    "BIOLINE USA": "BUS",
    "BIOLINE AGROSCIENCES MEXICANA": "BUS",
}

## Ventes, depuis le modèle sémantique du dashboard commercial

In [ ]:
df_sales = fabric.evaluate_measure(
    dataset=DS,
    workspace=WS,
    measure=[SALES_MEASURE],
    groupby_columns=["d_Entity[Entity]", "d_Calendar[Year]", "d_Calendar[Month]"],
    filters={"d_Scenary[Cod_Scenary]": ["1"], "d_Calendar[Year]": list(range(FIRST_YEAR, 2101))},
)

In [ ]:
# Une ligne par (entité SAP, année, mois), en € et en k€
sales = pd.DataFrame(df_sales).rename(columns={SALES_MEASURE: "Sales_€"})
sales["BU"] = sales["Entity"].map(ENTITY_TO_BU)
unmapped = sales.loc[sales["BU"].isna(), "Entity"].unique()
if len(unmapped):
    print(f"WARNING: entité(s) SAP sans BU, ignorées (à ajouter dans ENTITY_TO_BU) : {list(unmapped)}")
sales = sales.dropna(subset=["BU"])
sales["Year"] = sales["Year"].astype(int)
sales["Month"] = sales["Month"].astype(int)
sales["Sales_€"] = sales["Sales_€"].astype(float)
sales["Env.1"] = sales["Sales_€"] / 1000

sales_long = sales.melt(id_vars=["Entity", "BU", "Year", "Month"], value_vars=["Env.1", "Sales_€"],
                        var_name="ID", value_name="Value")

# Contrôle visuel : ventes en k€ par BU et par mois
print(sales.pivot_table(index="BU", columns="Month", values="Env.1", aggfunc="sum").round(0).to_string())

## Indicateurs RSE

In [ ]:
def read_csr_table(name: str) -> pd.DataFrame:
    """Lit une table écrite par nb_consolidate_csr_data (Month = nom du
    mois) et ajoute Month en numéro + MonthName."""
    df = spark.read.table(name).toPandas()
    if "Year" not in df.columns:
        # Les tables écrites par nb_consolidate_csr_data ont toujours Year :
        # sans elle, c'est encore l'ancienne version produite par le Dataflow.
        print(f"WARNING: {name} n'a pas de colonne Year — vérifier que le Dataflow ne l'écrase plus. "
              f"Year = {FIRST_YEAR} par défaut.")
        df["Year"] = FIRST_YEAR
    df = df.rename(columns={"Month": "MonthName"})
    df["Month"] = df["MonthName"].map(lambda m: MONTHS.index(m) + 1)
    df["Year"] = df["Year"].astype(int)
    return df


def add_date(df: pd.DataFrame) -> pd.DataFrame:
    df["Date"] = pd.to_datetime(dict(year=df["Year"], month=df["Month"], day=1)).dt.date
    return df

In [ ]:
indicators = read_csr_table("CSR_indicators_report")[["BU", "Year", "Month", "ID", "Value"]]
indicators["Entity"] = None

reporting = pd.concat([indicators, sales_long], ignore_index=True).dropna(subset=["Value"])
reporting["Value"] = reporting["Value"].astype(float)
reporting["MonthName"] = reporting["Month"].map(lambda m: MONTHS[m - 1])
reporting = add_date(reporting)
print(f"{REPORTING_TABLE}: {len(reporting)} rows ({reporting.ID.isin(['Env.1', 'Sales_€']).sum()} sales).")

## Suivi de complétion

In [ ]:
tracking = add_date(read_csr_table("CSR_completion_report"))
print(f"{TRACKING_TABLE}: {len(tracking)} rows.")

## Écriture

In [ ]:
REPORTING_SCHEMA = StructType([
    StructField("Entity", StringType()), StructField("BU", StringType()),
    StructField("Year", LongType()), StructField("Month", LongType()),
    StructField("MonthName", StringType()), StructField("ID", StringType()),
    StructField("Value", DoubleType()), StructField("Date", DateType()),
])
TRACKING_SCHEMA = StructType([
    StructField("BU", StringType()), StructField("Year", LongType()),
    StructField("Month", LongType()), StructField("MonthName", StringType()),
    StructField("completion_percent", LongType()), StructField("Date", DateType()),
])


_CASTS = {StringType: str, LongType: int, DoubleType: float}


def write_table(df: pd.DataFrame, name: str, schema: StructType):
    """pandas -> Delta avec un schéma explicite (types stables pour Power BI)."""
    rows = [
        tuple(None if pd.isna(v) else _CASTS.get(type(f.dataType), lambda x: x)(v) for v, f in zip(rec, schema.fields))
        for rec in df[[f.name for f in schema.fields]].astype(object).itertuples(index=False)
    ]
    (spark.createDataFrame(rows, schema)
     .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
     .saveAsTable(name))
    print(f"{name}: {len(rows)} row(s) written.")


write_table(reporting, REPORTING_TABLE, REPORTING_SCHEMA)
write_table(tracking, TRACKING_TABLE, TRACKING_SCHEMA)